In [24]:
# %pip install xgboost

In [25]:
import pandas as pd
import numpy as np
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, log_loss, f1_score, confusion_matrix
from xgboost import XGBClassifier

In [26]:
ml_data = pd.read_csv("Dataset/features.csv", parse_dates=["match_date"])
ml_data = ml_data.sort_values(["match_date", "id"]).reset_index(drop=True)

In [27]:
feature_cols = [
    "home_recent_5", "away_recent_5", "form_diff",
    "home_win_rate", "away_win_rate",
    "home_avg_gf", "home_avg_ga", "away_avg_gf", "away_avg_ga",
    "home_position", "away_position", "position_diff",
]
print([c for c in feature_cols if c not in ml_data.columns])   

[]


In [28]:
#Chronological split of data into train and test sets
train = ml_data[ml_data.season_start_year <= 2015]
val   = ml_data[(ml_data.season_start_year >= 2016) & (ml_data.season_start_year <= 2018)]
test  = ml_data[ml_data.season_start_year >= 2019]

X_train, y_train = train[feature_cols], train["target_encoded"]
X_val,   y_val   = val[feature_cols],   val["target_encoded"]
X_test,  y_test  = test[feature_cols],  test["target_encoded"]

print(len(train), len(val), len(test))

5286 1092 1086


In [29]:
base_pred = np.full(len(y_val), 2)   # always predict Home win (class 2)
print("Always-Home accuracy:", accuracy_score(y_val, base_pred))

Always-Home accuracy: 0.47619047619047616


In [30]:
#Evaluation helper function

def evaluate(model, X, y):
    pred = model.predict(X)
    proba = model.predict_proba(X)
    return {
        "accuracy": accuracy_score(y, pred),
        "log_loss": log_loss(y, proba), #judge predicted  probabilities// lower better and punishes confident wrong predictions
        "macro_f1": f1_score(y, pred, average="macro"), #averages performance across H/D/A equally and expose models that ignore minority class( draw in this case)
    }

In [31]:
#Model 1 - Logistic Regression
logreg= Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

logreg.fit(X_train, y_train)
print("LogReg val:", evaluate(logreg, X_val, y_val))

LogReg val: {'accuracy': 0.5677655677655677, 'log_loss': 0.9445477309585671, 'macro_f1': 0.4121063803744218}


In [32]:
#random forest
rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=10, random_state=42) #lower values to avoid overfitting
rf.fit(X_train, y_train)
print("RF val:", evaluate(rf, X_val, y_val))

RF val: {'accuracy': 0.5842490842490843, 'log_loss': 0.9383608588340124, 'macro_f1': 0.4241261957408901}


In [33]:
#XG boost
xgb = XGBClassifier(
    n_estimators=300, max_depth=2, learning_rate=0.03,
                          subsample=0.8, colsample_bytree=0.8,
                          eval_metric="mlogloss", random_state=42
)
xgb.fit(X_train, y_train)
print("XGB val:", evaluate(xgb, X_val, y_val))

XGB val: {'accuracy': 0.5805860805860806, 'log_loss': 0.9343813061714172, 'macro_f1': 0.4211672473867596}


In [34]:
results = pd.DataFrame({
    "LogReg": evaluate(logreg, X_val, y_val),
    "RandomForest": evaluate(rf, X_val, y_val),
    "XGBoost": evaluate(xgb, X_val, y_val),
}).T
print(results.round(4))

              accuracy  log_loss  macro_f1
LogReg          0.5678    0.9445    0.4121
RandomForest    0.5842    0.9384    0.4241
XGBoost         0.5806    0.9344    0.4212


Random forest shows best evaluation, but all are close. 

In [35]:
#hyperparameter tuning for XGBoost
best = {"loss": 9, "params": None}

for depth in [2, 3, 4]:
    for lr in [0.03, 0.05, 0.1]:
        for n in [100, 200, 300]:
            m = XGBClassifier(n_estimators=n, max_depth=depth, learning_rate=lr,
                              subsample=0.8, colsample_bytree=0.8,
                              eval_metric="mlogloss", random_state=42)
            m.fit(X_train, y_train)
            loss = log_loss(y_val, m.predict_proba(X_val))
            if loss < best["loss"]:
                best = {"loss": loss, "params": (depth, lr, n)}

print(best)

{'loss': 0.9343813061714172, 'params': (2, 0.03, 300)}


In [36]:
final_model = xgb  
print("TEST:", evaluate(final_model, X_test, y_test))

pred = final_model.predict(X_test)
print(confusion_matrix(y_test, pred))

TEST: {'accuracy': 0.505524861878453, 'log_loss': 1.0072808265686035, 'macro_f1': 0.3665962029674826}
[[158   0 218]
 [ 67   0 188]
 [ 64   0 391]]


In [37]:
test_res = test.copy()
test_res["pred"] = final_model.predict(X_test)

# accuracy and home-prediction rate per season
print(test_res.groupby("season_start_year").apply(
    lambda g: pd.Series({
        "accuracy": (g.pred == g.target_encoded).mean(),
        "always_home%": (g.target_encoded == 2).mean(),
        "pred_home%": (g.pred == 2).mean(),
    })
).round(3))

                   accuracy  always_home%  pred_home%
season_start_year                                    
2019                  0.522         0.454       0.730
2020                  0.476         0.380       0.745
2021                  0.520         0.422       0.727


In [ ]:
xgb_tuned = XGBClassifier(
    n_estimators=300, max_depth=2, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="mlogloss", random_state=42
)
xgb_tuned.fit(X_train, y_train)

print(evaluate(xgb_tuned, X_val, y_val))  

proba = xgb_tuned.predict_proba(X_test)
test_res = test.copy()
test_res["p_away"], test_res["p_draw"], test_res["p_home"] = proba[:,0], proba[:,1], proba[:,2]

print(test_res.groupby("season_start_year")[["p_away","p_draw","p_home"]].mean().round(3))
print(test_res.groupby("season_start_year")["target_encoded"]
      .value_counts(normalize=True).unstack().round(3))

{'accuracy': 0.5805860805860806, 'log_loss': 0.9343813061714172, 'macro_f1': 0.4211672473867596}
                   p_away  p_draw  p_home
season_start_year                        
2019                0.288   0.256   0.456
2020                0.287   0.258   0.456
2021                0.296   0.253   0.451
target_encoded         0      1      2
season_start_year                     
2019               0.305  0.241  0.454
2020               0.394  0.226  0.380
2021               0.339  0.239  0.422


Year without any spectators (2020/21) diverges from the home advantage norm, excluding that year:

In [40]:
test_res["pred"] = xgb_tuned.predict(X_test)
test_res["correct"] = (test_res.pred == test_res.target_encoded)

print("All test seasons:", test_res.correct.mean().round(4))
print("Excluding 2020-21:", test_res[test_res.season_start_year != 2020].correct.mean().round(4))

All test seasons: 0.5055
Excluding 2020-21: 0.5209
